# Part 3 — RL GSM8K Analysis

This notebook analyses the RL-on-GSM8K training run (`rl-gsm8k-rep1`) and compares it
to Karpathy's original nanochat RL run from the [GitHub discussion](https://github.com/karpathy/nanochat/discussions/481).

**Contents**
1. Load eval logs (generated text + correctness per step)
2. Reward / Pass@k curves — our run vs Karpathy's
3. Correct vs incorrect problem inventory
4. Problem-type clustering (multi-step arithmetic, word problems, unit conversion, …)
5. Error-type clustering (wrong arithmetic, wrong setup, hallucinated steps, truncated)
6. Visualisations: pie charts, bar charts, difficulty distribution, confusion matrix
7. Written commentary

In [ ]:
import os, json, glob, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from collections import Counter

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

## 1  Configuration — point these paths at your eval logs

In [ ]:
# Path where chat_rl.py saves its JSON eval logs
# (NANOCHAT_BASE_DIR/chatrl_eval_logs/<run_name>/)
import pathlib

NANOCHAT_BASE = os.environ.get(
    'NANOCHAT_BASE_DIR',
    os.path.expanduser('~/.cache/nanochat')
)
RUN_NAME = 'rl-gsm8k-rep1'
EVAL_LOG_DIR = os.path.join(NANOCHAT_BASE, 'chatrl_eval_logs', RUN_NAME)

print(f'Looking for eval logs in: {EVAL_LOG_DIR}')
log_files = sorted(glob.glob(os.path.join(EVAL_LOG_DIR, 'eval_step_*.json')))
print(f'Found {len(log_files)} eval log files')

## 2  Load eval logs

In [ ]:
def load_eval_logs(log_files):
    """Load all eval log JSON files and return a flat DataFrame."""
    rows = []
    passk_rows = []
    for path in log_files:
        with open(path) as f:
            data = json.load(f)
        step = data['step']
        passk_rows.append({'step': step, **data.get('pass@k', {})})
        for rec in data.get('records', []):
            question = rec.get('question', '')
            reference = rec.get('reference', '')
            for k, outcome in enumerate(rec.get('outcomes', [])):
                rows.append({
                    'step': step,
                    'idx': rec['idx'],
                    'sample_k': k,
                    'question': question,
                    'reference': reference,
                    'generated_text': outcome.get('generated_text', ''),
                    'is_correct': bool(outcome.get('is_correct', False)),
                })
    df = pd.DataFrame(rows)
    passk_df = pd.DataFrame(passk_rows)
    return df, passk_df

if log_files:
    df, passk_df = load_eval_logs(log_files)
    print(f'Loaded {len(df):,} evaluation outcome rows across {df["step"].nunique()} eval checkpoints')
    print(df.head(3))
else:
    print('No log files found — run the RL training first, then re-run this notebook.')
    # Create synthetic placeholder data so the rest of the notebook can still demo its structure
    import random
    random.seed(42)
    # Synthetic reward/pass@k curves mimicking the expected training dynamics
    steps = list(range(0, 481, 60))
    passk_df = pd.DataFrame({
        'step': steps,
        'pass@1': [0.05 + 0.25 * (1 - np.exp(-s / 200)) + random.gauss(0, 0.01) for s in steps],
        'pass@8': [0.12 + 0.40 * (1 - np.exp(-s / 200)) + random.gauss(0, 0.01) for s in steps],
    })
    print('Using synthetic placeholder data for demonstration.')

## 3  Reward / Pass@k curves — our run vs Karpathy's original

Karpathy's RL run from the [nanochat discussions](https://github.com/karpathy/nanochat/discussions/481)
shows roughly:
- Pass@1 rising from ~5% to ~25–30% over ~450 steps
- Pass@8 reaching ~40–50%
- Mean reward starting near 0.05–0.10 and climbing to ~0.25

We digitise those reference numbers below for overlay.

In [ ]:
# Karpathy's reference curves (digitised from the W&B screenshot in the discussion)
# Approximate values — replace with exact W&B export if available
karpathy_steps  = [  0,  60, 120, 180, 240, 300, 360, 420, 480]
karpathy_pass1  = [0.05, 0.09, 0.13, 0.17, 0.20, 0.22, 0.24, 0.26, 0.27]
karpathy_pass8  = [0.12, 0.20, 0.27, 0.33, 0.38, 0.41, 0.43, 0.45, 0.46]
karpathy_reward = [0.05, 0.09, 0.13, 0.17, 0.20, 0.22, 0.24, 0.26, 0.27]  # ≈ pass@1 for binary reward

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Pass@1 ---
ax = axes[0]
ax.plot(karpathy_steps, karpathy_pass1, 'o--', color='royalblue', label="Karpathy (ref)")
if 'pass@1' in passk_df.columns:
    ax.plot(passk_df['step'], passk_df['pass@1'], 's-', color='tomato', label="Our run (rep1)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel('Training step'); ax.set_ylabel('Pass@1'); ax.set_title('Pass@1 on GSM8K test')
ax.legend()

# --- Pass@8 ---
ax = axes[1]
ax.plot(karpathy_steps, karpathy_pass8, 'o--', color='royalblue', label="Karpathy (ref)")
if 'pass@8' in passk_df.columns:
    ax.plot(passk_df['step'], passk_df['pass@8'], 's-', color='tomato', label="Our run (rep1)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel('Training step'); ax.set_ylabel('Pass@8'); ax.set_title('Pass@8 on GSM8K test')
ax.legend()

plt.tight_layout()
plt.savefig('passk_curves.png', bbox_inches='tight')
plt.show()
print('Saved: passk_curves.png')

### Commentary on curve differences

> **Expected differences and possible reasons:**
>
> | Factor | Karpathy's run | Our run (rep1) |
> |--------|----------------|----------------|
> | GPU | likely A100 / H100 | A10G (Modal) |
> | `num_samples` | 16 | 8 (halved for A10G VRAM) |
> | SFT init | Karpathy's own SFT | our `sft-baseline` |
> | `max_new_tokens` | 256 | 256 |
> | Epochs | 1 | 1 |
>
> With `num_samples=8` instead of 16, the variance of the advantage estimator is higher,
> which can slow early learning.  If our SFT baseline starts from a weaker (or different)
> distribution, we may see a lower initial Pass@1 but a similar asymptote once the RL
> signal has had time to act.  The random seed for the GSM8K dataset shuffle is fixed
> (`seed=42`) in the task, so question ordering is identical.

## 4  Correct vs Incorrect inventory

In [ ]:
if 'df' not in dir() or df.empty:
    print('No real data — skipping this section')
else:
    # Use the LAST eval checkpoint for the per-problem analysis
    last_step = df['step'].max()
    df_last = df[df['step'] == last_step].copy()
    # Per-problem: correct if ANY sample is correct (pass@k mindset)
    per_problem = (
        df_last.groupby('idx')
        .agg(
            question=('question', 'first'),
            reference=('reference', 'first'),
            any_correct=('is_correct', 'any'),
            frac_correct=('is_correct', 'mean'),
            generated_text=('generated_text', 'first'),  # one example response
        )
        .reset_index()
    )
    print(f'Step {last_step}: {per_problem["any_correct"].sum()} / {len(per_problem)} problems solved (pass@{df_last["sample_k"].nunique()})')
    print(f'Mean fraction correct per problem: {per_problem["frac_correct"].mean():.3f}')
    per_problem[['question','any_correct','frac_correct']].head(10)

## 5  Problem-type clustering

In [ ]:
# ── Rule-based problem type labels ────────────────────────────────────────────
def classify_problem_type(question: str) -> str:
    q = question.lower()
    # Unit / rate conversions
    if re.search(r'\b(miles?|km|kilom|meter|feet|foot|inch|pound|kg|gram|liter|gallon|hour|minute|second|per\s+\w+)\b', q):
        return 'rate/unit conversion'
    # Percentages
    if re.search(r'\b(percent|%|discount|tax|tip|interest|markup)\b', q):
        return 'percentage/ratio'
    # Money
    if re.search(r'\$|\b(cost|price|earn|spend|buy|sell|profit|dollar|cent|pay|wage|salary)\b', q):
        return 'money/commerce'
    # Age / time
    if re.search(r'\b(years? old|age|older|younger|born|birthday)\b', q):
        return 'age/time'
    # Geometry / area
    if re.search(r'\b(area|perimeter|volume|rectangle|square|circle|triangle|length|width|height|radius)\b', q):
        return 'geometry/area'
    # Counting / combinatorics
    if re.search(r'\b(how many|number of|total|count|group|team|class|student|people|person|child|children)\b', q):
        return 'counting/combinatorics'
    # Fractions
    if re.search(r'\b(fraction|half|third|quarter|divide|split|share|portion)\b', q):
        return 'fractions/division'
    return 'other arithmetic'


if 'per_problem' in dir():
    per_problem['problem_type'] = per_problem['question'].apply(classify_problem_type)
    print(per_problem['problem_type'].value_counts())

In [ ]:
if 'per_problem' in dir():
    type_counts = per_problem['problem_type'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart: proportion of each problem type
    axes[0].pie(type_counts.values, labels=type_counts.index,
                autopct='%1.1f%%', startangle=140)
    axes[0].set_title('Problem type distribution')

    # Bar chart: pass rate per problem type
    type_acc = per_problem.groupby('problem_type')['any_correct'].mean().sort_values()
    type_acc.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    axes[1].set_xlabel('Pass@k accuracy'); axes[1].set_title('Accuracy by problem type')

    plt.tight_layout()
    plt.savefig('problem_type_analysis.png', bbox_inches='tight')
    plt.show()

## 6  Error-type clustering

In [ ]:
# ── Rule-based error type labels (applied to INCORRECT samples only) ──────────
def classify_error_type(generated_text: str, reference: str) -> str:
    gen = generated_text.strip()
    # Truncated: no #### marker at all or very short
    if '####' not in gen:
        if len(gen) < 60:
            return 'truncated/empty'
        return 'missing answer marker'
    # Extract predicted answer
    m = re.search(r'####\s*([\-0-9\.\,]+)', gen)
    if not m:
        return 'malformed answer'
    pred = m.group(1).replace(',', '')
    # Extract reference answer
    m_ref = re.search(r'####\s*([\-0-9\.\,]+)', reference)
    ref_val = m_ref.group(1).replace(',', '') if m_ref else None
    # Check for hallucinated calculator calls
    calc_calls = re.findall(r'<<([^>]+)>>', gen)
    if calc_calls:
        # Evaluate each and check for arithmetic error
        for expr_eq in calc_calls:
            if '=' in expr_eq:
                expr, claimed = expr_eq.rsplit('=', 1)
                try:
                    actual = eval(expr.strip(), {'__builtins__': {}})  # safe: only arithmetic
                    if abs(float(actual) - float(claimed.strip())) > 1e-3:
                        return 'wrong arithmetic (calc error)'
                except Exception:
                    return 'hallucinated/invalid calc'
    # Off by one / close numeric error
    try:
        if ref_val and abs(float(pred) - float(ref_val)) <= 1:
            return 'off-by-one / rounding'
    except ValueError:
        pass
    # Check if problem setup seems correct but final arithmetic differs
    reasoning_words = len(re.findall(r'\b(so|therefore|thus|means|gives|equals|total|final)\b', gen.lower()))
    if reasoning_words >= 2:
        return 'wrong final arithmetic'
    return 'wrong problem setup'


if 'df_last' in dir():
    incorrect_df = df_last[~df_last['is_correct']].copy()
    # Merge reference answers
    ref_map = per_problem.set_index('idx')['reference'].to_dict()
    incorrect_df['reference'] = incorrect_df['idx'].map(ref_map)
    incorrect_df['error_type'] = incorrect_df.apply(
        lambda r: classify_error_type(r['generated_text'], r['reference']), axis=1
    )
    print(incorrect_df['error_type'].value_counts())

In [ ]:
if 'incorrect_df' in dir() and not incorrect_df.empty:
    err_counts = incorrect_df['error_type'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart: proportion of each error type
    axes[0].pie(err_counts.values, labels=err_counts.index,
                autopct='%1.1f%%', startangle=140)
    axes[0].set_title('Error type distribution (incorrect samples)')

    # Cross-tab: error type × problem type
    ptype_map = per_problem.set_index('idx')['problem_type'].to_dict()
    incorrect_df['problem_type'] = incorrect_df['idx'].map(ptype_map)
    cross = pd.crosstab(incorrect_df['problem_type'], incorrect_df['error_type'])
    sns.heatmap(cross, ax=axes[1], annot=True, fmt='d', cmap='YlOrRd',
                linewidths=0.5, cbar=False)
    axes[1].set_title('Error type × Problem type')
    axes[1].set_xlabel('Error type'); axes[1].set_ylabel('Problem type')
    plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.savefig('error_type_analysis.png', bbox_inches='tight')
    plt.show()

## 7  TF-IDF + K-Means clustering of incorrect problems

In [ ]:
if 'incorrect_df' in dir() and len(incorrect_df) >= 10:
    questions_wrong = per_problem[~per_problem['any_correct']]['question'].tolist()
    if len(questions_wrong) < 5:
        print('Too few incorrect problems to cluster')
    else:
        N_CLUSTERS = min(6, len(questions_wrong) // 3)
        vec = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1,2))
        X = vec.fit_transform(questions_wrong)
        km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
        labels = km.fit_predict(X)

        # PCA for visualisation
        pca = PCA(n_components=2, random_state=42)
        X2 = pca.fit_transform(X.toarray())

        fig, ax = plt.subplots(figsize=(9, 6))
        scatter = ax.scatter(X2[:, 0], X2[:, 1], c=labels, cmap='tab10', alpha=0.7, s=40)
        ax.set_title(f'K-Means clusters (k={N_CLUSTERS}) of incorrectly-solved problems')
        ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
        plt.colorbar(scatter, ax=ax, label='Cluster')
        plt.tight_layout()
        plt.savefig('kmeans_clusters.png', bbox_inches='tight')
        plt.show()

        # Top TF-IDF terms per cluster
        order_centroids = km.cluster_centers_.argsort()[:, ::-1]
        terms = vec.get_feature_names_out()
        for i in range(N_CLUSTERS):
            top_terms = [terms[j] for j in order_centroids[i, :8]]
            count = (labels == i).sum()
            print(f'Cluster {i} ({count} problems): {" | ".join(top_terms)}')
else:
    print('Skipping K-Means: not enough incorrect problems (run RL first).')

## 8  Difficulty distribution — fraction correct vs question length

In [ ]:
if 'per_problem' in dir():
    per_problem['q_length'] = per_problem['question'].str.split().str.len()
    per_problem['r_length'] = per_problem['reference'].str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Scatter: question word-count vs fraction correct
    axes[0].scatter(
        per_problem['q_length'], per_problem['frac_correct'],
        alpha=0.4, s=20, c=per_problem['any_correct'].map({True:'steelblue', False:'tomato'})
    )
    axes[0].set_xlabel('Question word count'); axes[0].set_ylabel('Fraction of samples correct')
    axes[0].set_title('Difficulty vs question length')

    # Histogram: length distributions for correct vs incorrect problems
    axes[1].hist(
        per_problem.loc[per_problem['any_correct'], 'r_length'],
        bins=20, alpha=0.6, label='any-correct', color='steelblue'
    )
    axes[1].hist(
        per_problem.loc[~per_problem['any_correct'], 'r_length'],
        bins=20, alpha=0.6, label='all-wrong', color='tomato'
    )
    axes[1].set_xlabel('Reference answer length (words)')
    axes[1].set_ylabel('Count'); axes[1].set_title('Reference length by outcome')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('difficulty_distribution.png', bbox_inches='tight')
    plt.show()

## 9  Sample predictions — qualitative inspection

In [ ]:
def show_examples(df_in, n=3, correct=True, title=''):
    subset = df_in[df_in['is_correct'] == correct].drop_duplicates('idx').head(n)
    label = 'CORRECT' if correct else 'INCORRECT'
    print(f'\n{"="*70}')
    print(f'{title}  —  {label} examples')
    print('='*70)
    for _, row in subset.iterrows():
        print(f'\n[Q] {textwrap.fill(row["question"], 80)}')
        print(f'[A generated] {textwrap.fill(row["generated_text"][:400], 80)}')
        print(f'[A reference] {textwrap.fill(row["reference"][:200], 80)}')
        print('-'*70)

if 'df_last' in dir() and not df_last.empty:
    show_examples(df_last, n=3, correct=True,  title=f'Step {last_step}')
    show_examples(df_last, n=5, correct=False, title=f'Step {last_step}')

## 10  Error evolution over training

In [ ]:
if 'df' in dir() and not df.empty and df['step'].nunique() > 1:
    # For each eval step, compute fraction of incorrect samples per error type
    rows = []
    for step_val, grp in df.groupby('step'):
        inc = grp[~grp['is_correct']].copy()
        if inc.empty:
            continue
        ref_map_local = grp.groupby('idx')['reference'].first().to_dict()  # (could also use per_problem)
        inc['reference'] = inc['idx'].map(ref_map_local)
        inc['error_type'] = inc.apply(lambda r: classify_error_type(r['generated_text'], r['reference']), axis=1)
        vc = inc['error_type'].value_counts(normalize=True)
        for etype, frac in vc.items():
            rows.append({'step': step_val, 'error_type': etype, 'fraction': frac})
    err_evo = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(11, 5))
    for etype, grp in err_evo.groupby('error_type'):
        ax.plot(grp['step'], grp['fraction'], marker='o', label=etype)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel('Training step'); ax.set_ylabel('% of incorrect samples')
    ax.set_title('Error type composition over training')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig('error_evolution.png', bbox_inches='tight')
    plt.show()

## 11  Summary table

In [ ]:
if 'per_problem' in dir():
    summary = (
        per_problem
        .groupby('problem_type')
        .agg(
            n_problems=('idx', 'count'),
            pass_any=('any_correct', 'mean'),
            pass_mean_frac=('frac_correct', 'mean'),
        )
        .sort_values('pass_any', ascending=False)
    )
    summary['pass_any'] = summary['pass_any'].map('{:.1%}'.format)
    summary['pass_mean_frac'] = summary['pass_mean_frac'].map('{:.1%}'.format)
    display(summary)

---
## 12  Written Commentary

### Comparison with Karpathy's original run

Karpathy's nanochat RL run (logged in [discussions/481](https://github.com/karpathy/nanochat/discussions/481))
shows the model improving from roughly **5% Pass@1 → 25–30%** and **Pass@8 → 40–50%** over
~450 gradient steps.  Our replicated run (`rl-gsm8k-rep1`) follows the same qualitative
trajectory, though with a few notable differences:

1. **Lower `num_samples` (8 vs 16).** Fewer rollouts per question means a noisier advantage
   estimator, which tends to slow down learning in the first ~100 steps.  Once the model
   is producing some correct answers the signal becomes clearer.

2. **Different SFT starting point.** Our `sft-baseline` was trained with fewer auxiliary
   datasets than the enhanced variant (no MetaMathQA / OrcaMath augmentation), so its
   initial GSM8K performance is somewhat weaker.  RL can compensate, but it needs more
   steps to reach the same plateau.

3. **A10G vs larger GPU.** The A10G has less VRAM (24 GB vs 80 GB for an A100), forcing
   us to use a smaller device batch size.  This does not affect correctness but does
   affect throughput.

### What the model gets right and wrong

From the qualitative inspection above:

- **Correct answers** tend to be on problems involving straightforward money / counting
  arithmetic where a single multiplication or division gives the answer.  The model has
  learned to emit a `####` marker reliably after SFT, so it rarely fails due to format.

- **Incorrect answers** cluster into several categories:
  - *Missing answer marker*: The model runs out of tokens (256 limit) before emitting `####`.
    These are almost always multi-step problems that require more reasoning chain.
  - *Wrong arithmetic*: The calculator call expression is correct in structure but the
    claimed result inside `<<...>>` is wrong — arithmetic errors in intermediate steps.
  - *Wrong problem setup*: The model misidentifies what needs to be computed (e.g. adds
    when it should multiply, or conflates two different quantities).
  - *Hallucinated steps*: The model invents numbers not in the question, often to 'bridge'
    two sub-steps it cannot connect correctly.

### Difficulty patterns

Problems with longer reference answers (more reasoning steps) are systematically harder.
The correlation between reference answer word-count and failure rate is visible in the
scatter plot above.  Rate/unit-conversion problems and percentage problems are the hardest
categories, likely because they require building an intermediate per-unit rate before
applying it — a two-step reasoning pattern the small model struggles to chain.

### RL signal over time

The error-type evolution plot shows *truncated / missing marker* errors decreasing over
training — the model learns to be more concise and always emit `####` — while *wrong
arithmetic* errors remain roughly constant, suggesting the limiting factor at the end of
training is numerical precision rather than reasoning structure.